# Stochastic Calculus, Step by Step · Lesson 19: The Black-Scholes equation

Companion notebook for the lesson. Every number in the lesson's text, tables and figures is produced by running this notebook and read from `build/outputs.json`.

> **The course's numbers.** A share priced at S₀ = 100, a strike K = 105, a risk-free rate r = 3% (continuously compounded), a volatility σ = 20% and a horizon T = 1 year. Where a lesson needs the share's real-world expected return, it is μ = 8%. Made up for teaching; every random number is simulated with a fixed seed.
>
> Educational material only, not financial advice.

1. Setup · 2. A self-financing portfolio · 3. Itô's lemma on the option · 4. Matching the dW terms: the hedge · 5. Matching the dt terms: the equation · 6. The expected return disappears · 7. Exercises · 8. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S0, K, r, sigma, T = 100.0, 105.0, 0.03, 0.20, 1.0     # the course's numbers
N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))   # standard normal distribution function
out = {"meta": {
    "series": "Stochastic Calculus, Step by Step",
    "lesson": 19,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}, "inputs": {"S0": S0, "K": K, "r": r, "sigma": sigma, "T": T, "r_pct": r * 100, "sigma_pct": sigma * 100}}

In [ ]:
mu = 0.08                         # the share's real-world expected return
theta_mpr = (mu - r) / sigma      # the market price of risk
Nv = np.vectorize(N)              # N, for arrays
phi = lambda x: np.exp(-np.asarray(x, float) ** 2 / 2) / math.sqrt(2 * math.pi)   # the normal density


def d_plus(s, tau, sig=sigma, k=K):
    return (np.log(s / k) + (r + 0.5 * sig**2) * tau) / (sig * np.sqrt(tau))


def d_minus(s, tau, sig=sigma, k=K):
    return d_plus(s, tau, sig, k) - sig * np.sqrt(tau)


def call(s, tau, sig=sigma, k=K):
    """The Black-Scholes call price: Lesson 20 derives it."""
    return s * Nv(d_plus(s, tau, sig, k)) - k * np.exp(-r * tau) * Nv(d_minus(s, tau, sig, k))


def put(s, tau, sig=sigma, k=K):
    return call(s, tau, sig, k) - s + k * np.exp(-r * tau)


out["bs"] = {"mu": mu, "theta": theta_mpr, "d_plus": float(d_plus(S0, T)), "d_minus": float(d_minus(S0, T)),
             "N_plus": N(float(d_plus(S0, T))), "N_minus": N(float(d_minus(S0, T))),
             "call": float(call(S0, T)), "put": float(put(S0, T))}
out["bs"]

## 2. A self-financing portfolio

Hold $\Delta_t$ shares and put the rest of your wealth $X_t$ in cash at rate $r$. Nothing is added or taken out, so over a short step

$$dX_t = \Delta_t\,dS_t + r\big(X_t - \Delta_tS_t\big)\,dt .$$

One day, with 0.5 shares and wealth 10, in numbers:

In [ ]:
rng = np.random.default_rng(seed)
dt_day = 1 / 252
X0_, D0_ = 10.0, 0.5
Z_ = rng.standard_normal(200_000)
S1_ = S0 * np.exp((mu - 0.5 * sigma**2) * dt_day + sigma * math.sqrt(dt_day) * Z_)
X1_ = D0_ * S1_ + (X0_ - D0_ * S0) * math.exp(r * dt_day)
out["port"] = {"X0": X0_, "D0": D0_, "cash": X0_ - D0_ * S0, "mean": float(X1_.mean() - X0_),
               "mean_rule": D0_ * mu * S0 * dt_day + r * (X0_ - D0_ * S0) * dt_day,
               "sd": float(X1_.std()), "sd_rule": D0_ * sigma * S0 * math.sqrt(dt_day)}
out["port"]

## 3. Itô's lemma on the option

Let the option be worth $V(t, S_t)$. With $dS = \mu S\,dt + \sigma S\,dW$, Itô's lemma (Lesson 16) gives

$$dV = \Big(V_t + \mu SV_S + \tfrac12\sigma^2S^2V_{SS}\Big)dt + \sigma SV_S\,dW .$$

Take the call at $S = 100$ with one year left. The partial derivatives come from small bumps of the formula, and the mean and spread of the one-day change agree with the lemma:

In [ ]:
h_s, h_t = 0.01, 1e-5
v0 = float(call(S0, T))
V_S = float((call(S0 + h_s, T) - call(S0 - h_s, T)) / (2 * h_s))
V_SS = float((call(S0 + h_s, T) - 2 * call(S0, T) + call(S0 - h_s, T)) / h_s**2)
V_t = float((call(S0, T - h_t) - call(S0, T + h_t)) / (2 * h_t))     # calendar time up means tau down
V1_ = call(S1_, T - dt_day)
ito = {"V": v0, "V_S": V_S, "V_SS": V_SS, "V_t": V_t,
       "drift": V_t + mu * S0 * V_S + 0.5 * sigma**2 * S0**2 * V_SS, "vol": sigma * S0 * V_S}
ito.update({"mean_sim": float((V1_ - v0).mean()), "mean_rule": ito["drift"] * dt_day, "sd_sim": float(V1_.std()),
            "sd_rule": ito["vol"] * math.sqrt(dt_day)})
out["ito"] = ito
ito

## 4. Matching the dW terms: the hedge

Copy the option with a self-financing portfolio, so $X_t = V(t, S_t)$ at all times. The two random terms must agree: $\Delta\,\sigma S\,dW = \sigma SV_S\,dW$, so

$$\Delta_t = V_S(t, S_t) .$$

Over one day, holding $V_S$ shares removes almost all of the risk: the option's one-day change has a large spread, and the hedged position's is tiny.

In [ ]:
hedged = (V1_ - v0) - V_S * (S1_ - S0)
out["hedge"] = {"delta": V_S, "sd_option": float(V1_.std()), "sd_hedged": float(hedged.std()),
                "ratio": float(V1_.std() / hedged.std())}
sd_rows = []
for days in (21, 5, 1, 0.25):
    dt_ = days / 252
    Sx = S0 * np.exp((mu - 0.5 * sigma**2) * dt_ + sigma * math.sqrt(dt_) * Z_)
    Vx = call(Sx, T - dt_)
    sd_rows.append({"days": days, "sd_option": float(Vx.std()), "sd_hedged": float(((Vx - v0) - V_S * (Sx - S0)).std())})
out["hedge"]["rows"] = sd_rows
pd.DataFrame(sd_rows)

## 5. Matching the dt terms: the equation

With $\Delta = V_S$, what is left of $dX = dV$ is the $dt$ terms. The $\mu SV_S$ on both sides cancels, leaving the **Black-Scholes equation**

$$V_t + rSV_S + \tfrac12\sigma^2S^2V_{SS} = rV .$$

Each term at $S = 100$ with one year left, and the size of what's left over across a grid of prices and times:

In [ ]:
terms = {"theta": V_t, "r_S_delta": r * S0 * V_S, "half_gamma": 0.5 * sigma**2 * S0**2 * V_SS, "rV": r * v0}
terms["lhs"] = terms["theta"] + terms["r_S_delta"] + terms["half_gamma"]
terms["gap"] = terms["lhs"] - terms["rV"]


def residual(f, s, tau, h=0.01, k=1e-5):
    # V_t + r S V_S + 0.5 sigma^2 S^2 V_SS - r V, by small bumps; t up means tau down.
    Vt = (f(s, tau - k) - f(s, tau + k)) / (2 * k)
    Vs = (f(s + h, tau) - f(s - h, tau)) / (2 * h)
    Vss = (f(s + h, tau) - 2 * f(s, tau) + f(s - h, tau)) / h**2
    return Vt + r * s * Vs + 0.5 * sigma**2 * s**2 * Vss - r * f(s, tau)


grid_s, grid_t = np.array([70.0, 85.0, 100.0, 115.0, 130.0]), np.array([0.25, 0.5, 1.0])
res_call = max(abs(float(residual(call, s, t_))) for s in grid_s for t_ in grid_t)
res_put = max(abs(float(residual(put, s, t_))) for s in grid_s for t_ in grid_t)
share = lambda s, tau: s
cash_ = lambda s, tau: K * np.exp(-r * tau)
square = lambda s, tau: s**2 * np.exp((r + sigma**2) * tau)
wrong = lambda s, tau: s**2 * np.exp(r * tau)
out["pde"] = {"terms": terms, "res_call": res_call, "res_put": res_put, "res_share": abs(float(residual(share, 100.0, 0.5))),
              "res_cash": abs(float(residual(cash_, 100.0, 0.5))), "res_square": abs(float(residual(square, 100.0, 0.5))),
              "res_wrong": abs(float(residual(wrong, 100.0, 0.5)))}
out["pde"]

## 6. The expected return disappears

$\mu$ is nowhere in the equation. So the hedge doesn't care what the share is expected to earn: the one-day hedged position has the same spread, and on average earns the same (cash rate plus a tiny remainder), whatever $\mu$ is. Below, $\mu$ runs from 0 to 40%.

In [ ]:
mu_rows = []
for m in (0.0, 0.08, 0.2, 0.4):
    Sm = S0 * np.exp((m - 0.5 * sigma**2) * dt_day + sigma * math.sqrt(dt_day) * Z_)
    Vm = call(Sm, T - dt_day)
    pnl = (Vm - v0) - V_S * (Sm - S0)
    mu_rows.append({"mu_pct": m * 100, "option_mean": float((Vm - v0).mean()), "hedged_mean": float(pnl.mean()),
                    "hedged_sd": float(pnl.std())})
out["mu"] = {"rows": mu_rows, "cash_earned": r * (v0 - V_S * S0) * dt_day}
pd.DataFrame(mu_rows)

In [ ]:
# Chart data for the video (no random numbers)
sg = np.linspace(70.0, 130.0, 121)
out["hedge"]["fig"] = {"S": sg.tolist(), "V": call(sg, T).tolist(), "tangent": (out["ito"]["V"] + out["hedge"]["delta"] * (sg - S0)).tolist(),
                       "ticks": [{"x": float(x), "label": f"{x:.0f}"} for x in (70, 85, 100, 115, 130)]}

## 7. Exercises

Try each one before looking anything up. Full solutions are in the companion volume, *Part 4: Solutions*.

**Exercise 1.** A portfolio holds $\Delta_t$ shares and the rest in cash at rate $r$. Derive $dX_t = \Delta_t\,dS_t + r(X_t - \Delta_tS_t)\,dt$, then show with the product rule that $d(e^{-rt}X_t) = \Delta_t\,d(e^{-rt}S_t)$.

**Exercise 2.** Apply Itô's lemma to $V(t, S_t)$ with $dS = \mu S\,dt + \sigma S\,dW$, writing out each term of the Taylor expansion and showing why the others vanish.

**Exercise 3.** Derive the Black-Scholes equation a second way: form $\Pi = V - V_SS$, with $V_S$ held fixed over the step, show that $d\Pi$ has no $dW$ term, and argue that $\Pi$ must earn the risk-free rate.

**Exercise 4.** Check that $V = S$ and $V = Ke^{-r(T - t)}$ both solve the equation. What contract does their difference describe, and why must it solve the equation too?

**Exercise 5.** Find the constant $b$ for which $V = S^2e^{b(T - t)}$ solves the equation. A contract pays $S_T^2$ at $T$: what is it worth today with the course's numbers?

**Exercise 6.** A share pays a continuous dividend yield $q$: holding it pays $qS\,dt$ in cash. Rewrite the self-financing equation and show that the Black-Scholes equation becomes $V_t + (r - q)SV_S + \tfrac12\sigma^2S^2V_{SS} = rV$.

**Exercise 7.** (Code) Use small bumps of the formula to check the equation for the call and the put at $S = 70, 85, 100, 115, 130$ and $T - t = 0.25, 0.5, 1$. What is the largest residual? Then try the payoff-shaped function $V = \max(S - K, 0)$ at $S = 100$ and $S = 110$: which terms fail?

**Exercise 8.** (Code) For the call with one year left, simulate the hedged change $V(t + h, S_{t+h}) - V(t, S_t) - V_S(S_{t+h} - S_t)$ with $h$ of 16, 4 and 1 trading days. By what factor does its standard deviation shrink each time $h$ is divided by 4, and why?

## 8. Export

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")